# Mockture: Flow Composer Widget

The `mockture.widgets` module provides an interactive Jupyter widget for composing and
saving mockture flows visually. Instead of writing YAML by hand, you pick interactions
from a template library, fill in argument values, inspect the result as a live Mermaid
sequence diagram, and save the composed flow back to a flows YAML file.

After completing this notebook you will be able to:

- Render any list of resolved interactions as a Mermaid sequence diagram using
  `flow_to_mermaid`.
- Launch a `FlowComposer` widget, add and reorder interactions, edit template args
  in-place, and watch the diagram update live.
- Save a named flow to a flows YAML file directly from the widget.
- Load and inspect the saved flow programmatically.

## Table of Contents

- [Prerequisites](#Prerequisites)
- [1. Setup](#1-Setup)
  - [1.1 Install Dependencies](#11-Install-Dependencies)
  - [1.2 Import Packages](#12-Import-Packages)
- [2. Mermaid Export](#2-Mermaid-Export)
  - [2.1 flow_to_mermaid](#21-flow_to_mermaid)
  - [2.2 Render in Notebook](#22-Render-in-Notebook)
- [3. Flow Composer Widget](#3-Flow-Composer-Widget)
  - [3.1 Launch the Composer](#31-Launch-the-Composer)
  - [3.2 Save a Flow](#32-Save-a-Flow)
- [4. Load the Saved Flow](#4-Load-the-Saved-Flow)
- [5. Conclusion](#5-Conclusion)

## Prerequisites

- `mockture` installed in the active environment.
- `ipywidgets` installed (included in the `[widgets]` extra).
- The `example/01_basic/configs/` directory present in the repository — this notebook
  borrows its template and OpenAPI files so no new fixtures are required.
- JupyterLab 3+ or classic Jupyter Notebook. The Mermaid diagram preview uses an ES
  module CDN import and requires an internet connection on first render.

## 1. Setup

### 1.1 Install Dependencies

Install the `[widgets]` extra which pulls in `ipywidgets` and `ipython`.

In [1]:
%pip install -e .[widgets]

Obtaining file:///C:/Users/acisse/Documents/CodeWorkspace/mockture/usage
Note: you may need to restart the kernel to use updated packages.


ERROR: file:///C:/Users/acisse/Documents/CodeWorkspace/mockture/usage does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


### 1.2 Import Packages

In [ ]:
import sys
from pathlib import Path

import yaml
from IPython.display import HTML, display

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

from mockture.widgets import FlowComposer, flow_to_mermaid, mermaid_html

# Paths — reuse the basic example fixtures so no new files are needed
repo_root      = Path.cwd().parent
templates_path = repo_root / "example" / "01_basic" / "configs" / "basic_api.templates.yml"
flows_path     = Path("composed.flows.yml")

print("templates_path:", templates_path)
print("flows_path:    ", flows_path)

ModuleNotFoundError: No module named 'mockture.widgets'

## 2. Mermaid Export

### 2.1 flow_to_mermaid

`flow_to_mermaid` is a pure function that converts a list of rendered interaction dicts
to a Mermaid `sequenceDiagram` string. Each dict must contain `method`, `path`, and
`response` keys — the same shape returned by `TemplateLibrary.render()`.

You can call it standalone without any widget infrastructure.

In [ ]:
steps = [
    {
        "method": "POST",
        "path": "/orders",
        "response": {"status": 201, "body": {"order_id": "ord-1", "status": "created"}},
    },
    {
        "method": "GET",
        "path": "/orders/ord-1",
        "response": {"status": 200, "body": {"order_id": "ord-1", "status": "created"}},
    },
    {
        "method": "POST",
        "path": "/orders",
        "response": {"status": 409, "body": {"message": "Item is unavailable"}},
    },
]

diagram = flow_to_mermaid(steps)
print(diagram)

### 2.2 Render in Notebook

`mermaid_html` wraps the diagram string in self-contained HTML that loads Mermaid from
the CDN. Pass the result to `IPython.display.HTML` to render it inline.

In [ ]:
display(HTML(mermaid_html(diagram)))

## 3. Flow Composer Widget

### 3.1 Launch the Composer

`FlowComposer` loads a `TemplateLibrary` from `templates_path` and builds an ipywidgets
interface. Call `.show()` to render it in the current cell.

**Controls:**

- **Template dropdown + Add** — pick a template and click Add to append it to the flow.
  The interaction is pre-filled with template defaults. Required args (marked `*`) must
  be set before saving.
- **Arg text fields** — edit any argument inline. The sequence diagram re-renders on
  every keystroke.
- **Arrow buttons** — reorder steps up or down.
- **Trash button** — remove a step.
- **Flow name + Save** — type a name and click Save to write the flow to `flows_path`.

In [ ]:
composer = FlowComposer(
    templates_path=str(templates_path),
    flows_path=str(flows_path),
)
composer.show()

### 3.2 Save a Flow

The cell below demonstrates saving a flow programmatically, bypassing the Save button.
This is useful for scripting or testing. It seeds the composer state with two steps and
writes the result to `flows_path`.

In [ ]:
# Seed the composer with a two-step flow
composer._steps = [
    {
        "template_name": "create_order_success",
        "args": {"status_code": "201", "order_id": "ord-001", "status": "queued"},
    },
    {
        "template_name": "get_order",
        "args": {"order_id": "ord-001", "status_code": "200", "status": "queued"},
    },
]
composer._name_input.value = "create_and_fetch"

# Trigger save (same as clicking the Save button)
composer._on_save(None)
print("Saved to:", flows_path)
print(flows_path.read_text(encoding="utf-8"))

## 4. Load the Saved Flow

The saved YAML follows the same `flows:` structure that `Mockture` reads via the pytest
plugin. We load it here to confirm the output is valid and inspect the steps.

In [ ]:
saved = yaml.safe_load(flows_path.read_text(encoding="utf-8"))
flow  = saved["flows"]["create_and_fetch"]

print(f"Flow has {len(flow)} steps:")
for i, step in enumerate(flow, 1):
    print(f"  {i}. {step['template']}  args={step['args']}")

The saved file can be passed directly to the pytest plugin via `flows_path=`:

```python
@pytest.mark.mockture(
    contract="configs/api.openapi.yml",
    templates="configs/api.templates.yml",
    flows_path="composed.flows.yml",
    flow="create_and_fetch",
)
def test_create_and_fetch(mockture):
    r1 = httpx.post(mockture.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1})
    r2 = httpx.get(mockture.url_for("/orders/ord-001"))
    assert r1.status_code == 201
    assert r2.status_code == 200
```

## 5. Conclusion

This notebook covered the `mockture.widgets` module:

- Used `flow_to_mermaid` as a standalone function to convert resolved interaction dicts
  to a Mermaid sequence diagram string.
- Rendered a diagram inline with `mermaid_html` and `IPython.display.HTML`.
- Launched a `FlowComposer` widget to pick templates, edit args, reorder steps, and
  preview changes as a live sequence diagram.
- Saved the composed flow to a flows YAML file and confirmed the output format.

The saved YAML is compatible with the mockture pytest plugin `flow=` and `flows_path=`
marker arguments. See [`usage/mockture_server_usage.ipynb`](mockture_server_usage.ipynb)
for the full `Mockture` server object API.